# Network Measurement Lab — RTT Analysis

**Assignment**: Measure Round Trip Time (RTT) to selected servers using ICMP Echo Request / Echo Reply packets implemented in Python.

---

## Overview
This notebook:
1. Implements a raw-socket ICMP ping in pure Python.
2. Sends **N probes** to each target server and records the RTT of every successful reply.
3. Computes summary statistics: **min, max, mean, standard deviation, and packet-loss %**.
4. Visualises the results with **line plots**, **box plots**, and a **CDF** per server.

> ⚠️ **Note**: Raw sockets require root privileges.  
> Google Colab runs as `root`, so no extra setup is needed.  
> On a local machine run the notebook with `sudo jupyter notebook`.

## 1 · Imports & Dependencies

In [ ]:
import os
import socket
import struct
import time
import select
import random
import statistics
from collections import defaultdict

# Install matplotlib if not available (e.g. minimal Colab kernel)
try:
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib', '--quiet'])
    import matplotlib
    import matplotlib.pyplot as plt

import numpy as np

matplotlib.rcParams['figure.dpi'] = 120
print('All imports successful.')

## 2 · ICMP Ping Implementation

The implementation follows **RFC 792** for ICMP Echo Request / Reply:

| Field     | Bits | Value (request) |
|-----------|------|-----------------|
| Type      | 8    | 8               |
| Code      | 8    | 0               |
| Checksum  | 16   | computed        |
| Identifier| 16   | PID (mod 65535) |
| Sequence  | 16   | incremented     |

In [ ]:
ICMP_ECHO_REQUEST = 8   # Type field for Echo Request
ICMP_ECHO_REPLY   = 0   # Type field for Echo Reply
DEFAULT_TIMEOUT   = 2   # seconds to wait for a reply
PAYLOAD           = b'NetworkMeasurementLab-RTT'  # fixed payload


def _checksum(data: bytes) -> int:
    """Compute the Internet checksum (RFC 1071) over *data*."""
    s = 0
    # Sum 16-bit words
    for i in range(0, len(data) - 1, 2):
        s += (data[i] << 8) + data[i + 1]
    # Handle an odd-length payload
    if len(data) % 2 != 0:
        s += data[-1] << 8
    # Fold 32-bit sum to 16 bits
    while s >> 16:
        s = (s & 0xFFFF) + (s >> 16)
    return ~s & 0xFFFF


def _build_icmp_packet(identifier: int, sequence: int) -> bytes:
    """Build an ICMP Echo Request packet."""
    # Placeholder checksum = 0
    header = struct.pack('!BBHHH', ICMP_ECHO_REQUEST, 0, 0, identifier, sequence)
    packet = header + PAYLOAD
    chk = _checksum(packet)
    # Rebuild with correct checksum
    header = struct.pack('!BBHHH', ICMP_ECHO_REQUEST, 0, chk, identifier, sequence)
    return header + PAYLOAD


def ping_once(dest_addr: str, sequence: int, timeout: float = DEFAULT_TIMEOUT) -> float | None:
    """
    Send a single ICMP Echo Request to *dest_addr* and return the RTT in
    milliseconds, or ``None`` if the packet was lost / an error occurred.
    """
    identifier = os.getpid() & 0xFFFF

    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_RAW, socket.IPPROTO_ICMP)
    except PermissionError:
        raise PermissionError(
            'Raw sockets require root privileges.\n'
            'Run Colab as-is (it already runs as root), or locally use:\n'
            '    sudo jupyter notebook'
        )

    packet = _build_icmp_packet(identifier, sequence)

    try:
        send_time = time.perf_counter()
        sock.sendto(packet, (dest_addr, 0))

        deadline = send_time + timeout
        while True:
            remaining = deadline - time.perf_counter()
            if remaining <= 0:
                return None  # timed out

            readable, _, _ = select.select([sock], [], [], remaining)
            if not readable:
                return None  # timed out

            recv_time = time.perf_counter()
            raw_packet, _ = sock.recvfrom(1024)

            # IP header is 20 bytes; ICMP header starts at byte 20
            icmp_type, icmp_code, _, recv_id, recv_seq = struct.unpack(
                '!BBHHH', raw_packet[20:28]
            )

            if icmp_type == ICMP_ECHO_REPLY and recv_id == identifier and recv_seq == sequence:
                return (recv_time - send_time) * 1000  # convert to ms
            # Otherwise it's a stray packet — keep waiting
    finally:
        sock.close()


print('ICMP ping helpers defined.')

## 3 · Configuration

Adjust `TARGET_SERVERS` and `NUM_PROBES` as needed.

In [ ]:
# --- Target servers -----------------------------------------------------------
TARGET_SERVERS = [
    'google.com',
    'cloudflare.com',
    'amazon.com',
    'github.com',
    'microsoft.com',
]

# --- Measurement parameters ---------------------------------------------------
NUM_PROBES   = 20    # number of ICMP echo requests per server
PROBE_DELAY  = 0.5  # seconds between consecutive probes (to avoid flooding)
TIMEOUT      = 2    # seconds to wait for each reply

print(f'Targets  : {TARGET_SERVERS}')
print(f'Probes   : {NUM_PROBES} per server')
print(f'Delay    : {PROBE_DELAY} s between probes')
print(f'Timeout  : {TIMEOUT} s')

## 4 · Run Measurements

In [ ]:
def resolve(hostname: str) -> str | None:
    """Return the IPv4 address for *hostname*, or None on failure."""
    try:
        return socket.gethostbyname(hostname)
    except socket.gaierror as exc:
        print(f'  [!] Cannot resolve {hostname}: {exc}')
        return None


# results[server] = list of RTT values in ms (None = packet lost)
results: dict[str, list] = {}

for server in TARGET_SERVERS:
    print(f'\nPinging {server} …')
    ip = resolve(server)
    if ip is None:
        results[server] = [None] * NUM_PROBES
        continue
    print(f'  Resolved to {ip}')

    rtts = []
    for seq in range(1, NUM_PROBES + 1):
        rtt = ping_once(ip, seq, timeout=TIMEOUT)
        status = f'{rtt:.2f} ms' if rtt is not None else 'timeout'
        print(f'  Seq {seq:>3}: {status}')
        rtts.append(rtt)
        time.sleep(PROBE_DELAY)

    results[server] = rtts

print('\n✓ Measurements complete.')

## 5 · Statistical Analysis

In [ ]:
def compute_stats(rtts: list) -> dict:
    """Return a dict with summary statistics for a list of RTT measurements."""
    sent      = len(rtts)
    received  = [r for r in rtts if r is not None]
    lost      = sent - len(received)
    loss_pct  = 100.0 * lost / sent if sent else 0.0

    if received:
        return {
            'sent'      : sent,
            'received'  : len(received),
            'lost'      : lost,
            'loss_%'    : round(loss_pct, 1),
            'min_ms'    : round(min(received), 3),
            'max_ms'    : round(max(received), 3),
            'mean_ms'   : round(statistics.mean(received), 3),
            'stdev_ms'  : round(statistics.stdev(received), 3) if len(received) > 1 else 0.0,
            'median_ms' : round(statistics.median(received), 3),
        }
    else:
        return {
            'sent'      : sent,
            'received'  : 0,
            'lost'      : lost,
            'loss_%'    : 100.0,
            'min_ms'    : None,
            'max_ms'    : None,
            'mean_ms'   : None,
            'stdev_ms'  : None,
            'median_ms' : None,
        }


stats = {server: compute_stats(rtts) for server, rtts in results.items()}

# Print table
header = f"{'Server':<18} {'Sent':>4} {'Rcvd':>4} {'Loss%':>6} "\
         f"{'Min(ms)':>9} {'Max(ms)':>9} {'Mean(ms)':>10} {'StDev(ms)':>10} {'Median(ms)':>11}"
def _fmt_ms(v, fmt='.3f'):
    """Format a numeric value or return 'N/A' if None."""
    return format(v, fmt) if v is not None else 'N/A'

print(header)
print('-' * len(header))
for server, s in stats.items():
    print(
        f"{server:<18} {s['sent']:>4} {s['received']:>4} {s['loss_%']:>6.1f} "
        f"{_fmt_ms(s['min_ms']):>9} {_fmt_ms(s['max_ms']):>9} {_fmt_ms(s['mean_ms']):>10} "
        f"{_fmt_ms(s['stdev_ms']):>10} {_fmt_ms(s['median_ms']):>11}"
    )

## 6 · Visualisation

### 6.1 · RTT per Probe (Line Plot)

In [ ]:
fig, axes = plt.subplots(
    len(TARGET_SERVERS), 1,
    figsize=(12, 3 * len(TARGET_SERVERS)),
    sharex=True
)

if len(TARGET_SERVERS) == 1:
    axes = [axes]

colors = plt.cm.tab10.colors

for ax, (server, rtts), color in zip(axes, results.items(), colors):
    seqs   = list(range(1, len(rtts) + 1))
    values = [r if r is not None else float('nan') for r in rtts]
    lost_x = [s for s, r in zip(seqs, rtts) if r is None]
    lost_y = [0] * len(lost_x)

    ax.plot(seqs, values, marker='o', linewidth=1.4, markersize=4,
            color=color, label='RTT')
    if lost_x:
        ax.scatter(lost_x, lost_y, marker='x', color='red', zorder=5,
                   s=60, label='Packet lost')
    ax.set_ylabel('RTT (ms)', fontsize=9)
    ax.set_title(server, fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Probe sequence number', fontsize=10)
fig.suptitle('RTT per Probe — All Servers', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('rtt_line_plots.png', bbox_inches='tight')
plt.show()
print('Saved: rtt_line_plots.png')

### 6.2 · RTT Distribution (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

data_for_box = [
    [r for r in rtts if r is not None]
    for rtts in results.values()
]
labels = list(results.keys())

bp = ax.boxplot(
    data_for_box,
    labels=labels,
    patch_artist=True,
    notch=False,
    medianprops=dict(color='black', linewidth=2),
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Server', fontsize=11)
ax.set_ylabel('RTT (ms)', fontsize=11)
ax.set_title('RTT Distribution — Box Plot', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('rtt_box_plot.png', bbox_inches='tight')
plt.show()
print('Saved: rtt_box_plot.png')

### 6.3 · Empirical CDF of RTT per Server

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for (server, rtts), color in zip(results.items(), colors):
    received = sorted(r for r in rtts if r is not None)
    if not received:
        continue
    n    = len(received)
    cdf  = [(i + 1) / n for i in range(n)]
    ax.step(received, cdf, where='post', color=color, linewidth=2, label=server)

ax.set_xlabel('RTT (ms)', fontsize=11)
ax.set_ylabel('CDF', fontsize=11)
ax.set_title('Empirical CDF of RTT — All Servers', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('rtt_cdf.png', bbox_inches='tight')
plt.show()
print('Saved: rtt_cdf.png')

### 6.4 · Mean RTT Comparison (Bar Chart with Error Bars)

In [ ]:
servers_with_data = [s for s, st in stats.items() if st['mean_ms'] is not None]
means  = [stats[s]['mean_ms']  for s in servers_with_data]
stdevs = [stats[s]['stdev_ms'] for s in servers_with_data]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(servers_with_data))
bars = ax.bar(
    x, means, yerr=stdevs,
    capsize=6, color=[colors[i] for i in range(len(servers_with_data))],
    alpha=0.8, edgecolor='black', linewidth=0.8
)
ax.set_xticks(x)
ax.set_xticklabels(servers_with_data, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Mean RTT (ms)', fontsize=11)
ax.set_title('Mean RTT ± StDev — All Servers', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{mean:.1f}',
        ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.savefig('rtt_mean_bar.png', bbox_inches='tight')
plt.show()
print('Saved: rtt_mean_bar.png')

### 6.5 · Packet Loss Summary

In [ ]:
loss_vals = [stats[s]['loss_%'] for s in TARGET_SERVERS]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(TARGET_SERVERS))
ax.bar(
    x, loss_vals,
    color=[colors[i] for i in range(len(TARGET_SERVERS))],
    alpha=0.8, edgecolor='black', linewidth=0.8
)
ax.set_xticks(x)
ax.set_xticklabels(TARGET_SERVERS, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Packet Loss (%)', fontsize=11)
ax.set_title('Packet Loss per Server', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

for xi, val in zip(x, loss_vals):
    ax.text(xi, val + 1, f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('rtt_packet_loss.png', bbox_inches='tight')
plt.show()
print('Saved: rtt_packet_loss.png')

## 7 · Summary

The cell below re-prints the statistics table and highlights the best/worst performers.

In [ ]:
valid = {s: st for s, st in stats.items() if st['mean_ms'] is not None}

if valid:
    best  = min(valid, key=lambda s: valid[s]['mean_ms'])
    worst = max(valid, key=lambda s: valid[s]['mean_ms'])
    print(f'Lowest  mean RTT : {best}  ({valid[best]["mean_ms"]} ms)')
    print(f'Highest mean RTT : {worst} ({valid[worst]["mean_ms"]} ms)')
else:
    print('No successful measurements.')

print('\n--- Full statistics table ---')
header = f"{'Server':<18} {'Sent':>4} {'Rcvd':>4} {'Loss%':>6} "\
         f"{'Min':>9} {'Max':>9} {'Mean':>10} {'StDev':>10} {'Median':>11}"
print(header)
print('-' * len(header))
for server, s in stats.items():
    print(
        f"{server:<18} {s['sent']:>4} {s['received']:>4} {s['loss_%']:>6.1f} "
        f"{_fmt_ms(s['min_ms']):>9} {_fmt_ms(s['max_ms']):>9} {_fmt_ms(s['mean_ms']):>10} "
        f"{_fmt_ms(s['stdev_ms']):>10} {_fmt_ms(s['median_ms']):>11}"
    )